# Decoding Strategies

How to convert model logits into text. Each strategy makes different quality/diversity tradeoffs.

**Interview questions:**
- "Beam search vs sampling — when to use which?"
- "What does temperature do mathematically?"
- "How does nucleus sampling work?"

---

## Key Formula

**Temperature scaling:**
$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$: argmax (greedy)
- $T = 1$: original distribution
- $T \to \infty$: uniform distribution

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

torch.manual_seed(42)

## 1. Build a Tiny Language Model

A character-level bigram+ model to demonstrate decoding strategies with visible outputs.

In [ ]:
class TinyLM(nn.Module):
    """Tiny transformer LM for demonstrating decoding strategies."""
    def __init__(self, vocab_size: int, d_model: int = 64, n_heads: int = 4, 
                 n_layers: int = 2, max_len: int = 128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4*d_model,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len
    
    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.head(self.ln_f(x))

# Simple char-level vocabulary
text = "the quick brown fox jumps over the lazy dog " * 100
text += "a quick brown cat leaps over the sleepy dog " * 50
text += "the fast red fox runs past the quiet cat " * 50

chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

def encode(s): return [char_to_idx[c] for c in s]
def decode(ids): return ''.join([idx_to_char[i] for i in ids])

print(f"Vocab size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Train
data = torch.tensor(encode(text), dtype=torch.long)
model = TinyLM(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

model.train()
block_size = 32
losses = []

for step in range(800):
    # Random chunk
    ix = torch.randint(0, len(data) - block_size - 1, (32,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.tight_layout()
plt.show()
print(f"Final loss: {losses[-1]:.3f}")

## 2. Greedy Decoding

Always pick the highest-probability token. Deterministic, fast, but repetitive.

In [ ]:
@torch.no_grad()
def greedy_decode(model, prompt_ids, max_new_tokens=60):
    """Greedy: always pick argmax."""
    model.eval()
    ids = prompt_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
    return ids

prompt = torch.tensor([encode("the ")], dtype=torch.long)
output = greedy_decode(model, prompt)
print(f"Greedy: '{decode(output[0].tolist())}'")
print("Note: greedy decoding tends to be repetitive.")

## 3. Temperature Scaling

In [ ]:
@torch.no_grad()
def sample_with_temperature(model, prompt_ids, max_new_tokens=60, temperature=1.0):
    """Sample from softmax(logits / temperature)."""
    model.eval()
    ids = prompt_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return ids

# Visualize temperature effect on a distribution
logits_example = torch.tensor([3.0, 2.0, 1.5, 0.5, 0.1, -0.5, -1.0, -2.0])
labels = [f'tok_{i}' for i in range(len(logits_example))]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
temps = [0.3, 0.7, 1.0, 2.0]
for ax, T in zip(axes, temps):
    probs = F.softmax(logits_example / T, dim=-1).numpy()
    ax.bar(labels, probs, color='steelblue')
    ax.set_title(f'T = {T}')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    entropy = -(probs * np.log(probs + 1e-10)).sum()
    ax.text(0.5, 0.9, f'H={entropy:.2f}', transform=ax.transAxes, ha='center')

plt.suptitle('Effect of Temperature on Probability Distribution', fontsize=13)
plt.tight_layout()
plt.show()

print("Low T: peaked (confident, less diverse) | High T: flat (uncertain, more diverse)")

In [ ]:
# Generate with different temperatures
prompt = torch.tensor([encode("the ")], dtype=torch.long)
for T in [0.3, 0.7, 1.0, 1.5]:
    output = sample_with_temperature(model, prompt, temperature=T)
    print(f"T={T:.1f}: '{decode(output[0].tolist())}'")

## 4. Top-k Sampling (Fan et al., 2018)

Only sample from the top $k$ most probable tokens. Zero out everything else.

In [ ]:
def top_k_filtering(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Set logits below the top-k threshold to -inf."""
    if k <= 0:
        return logits
    # Find the k-th largest value
    top_k_values, _ = torch.topk(logits, k, dim=-1)
    threshold = top_k_values[..., -1:]
    # Mask everything below threshold
    return logits.masked_fill(logits < threshold, float('-inf'))


@torch.no_grad()
def top_k_sample(model, prompt_ids, max_new_tokens=60, k=5, temperature=1.0):
    model.eval()
    ids = prompt_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])[:, -1, :] / temperature
        logits = top_k_filtering(logits, k)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return ids

# Visualize top-k filtering
logits_ex = torch.tensor([3.0, 2.0, 1.5, 0.5, 0.1, -0.5, -1.0, -2.0])
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, [2, 4, 8]):
    filtered = top_k_filtering(logits_ex.unsqueeze(0), k).squeeze()
    probs = F.softmax(filtered, dim=-1).numpy()
    colors = ['steelblue' if p > 0 else 'lightgray' for p in probs]
    ax.bar(labels, probs, color=colors)
    ax.set_title(f'Top-k = {k}')
    ax.set_ylim(0, 0.8)
    ax.set_ylabel('Probability')

plt.suptitle('Top-k Sampling: Filtered Probability Distributions', fontsize=13)
plt.tight_layout()
plt.show()

# Generate
prompt = torch.tensor([encode("the ")], dtype=torch.long)
for k in [3, 5, 10]:
    output = top_k_sample(model, prompt, k=k)
    print(f"top-k={k}: '{decode(output[0].tolist())}'")

## 5. Nucleus (Top-p) Sampling (Holtzman et al., 2020)

Sample from the smallest set of tokens whose cumulative probability exceeds $p$.

Advantage over top-k: adapts the number of candidates to the distribution shape.
- Confident prediction: only 2-3 tokens might sum to $p=0.9$
- Uncertain prediction: might need 50+ tokens

In [ ]:
def top_p_filtering(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Nucleus sampling: keep smallest set of tokens with cumulative prob >= p."""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    # Find where cumulative prob exceeds p
    # Shift right so we keep the token that pushes us over p
    sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= p
    
    # Scatter the mask back to original indices
    sorted_logits[sorted_mask] = float('-inf')
    
    # Un-sort
    output = torch.zeros_like(logits)
    output.scatter_(dim=-1, index=sorted_indices, src=sorted_logits)
    return output


@torch.no_grad()
def top_p_sample(model, prompt_ids, max_new_tokens=60, p=0.9, temperature=1.0):
    model.eval()
    ids = prompt_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])[:, -1, :] / temperature
        logits = top_p_filtering(logits, p)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return ids

# Visualize top-p
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, p in zip(axes, [0.5, 0.8, 0.95]):
    filtered = top_p_filtering(logits_ex.unsqueeze(0), p).squeeze()
    probs = F.softmax(filtered, dim=-1).numpy()
    colors = ['steelblue' if pr > 0.001 else 'lightgray' for pr in probs]
    ax.bar(labels, probs, color=colors)
    n_kept = sum(1 for pr in probs if pr > 0.001)
    ax.set_title(f'Top-p = {p} ({n_kept} tokens kept)')
    ax.set_ylim(0, 0.8)
    ax.set_ylabel('Probability')

plt.suptitle('Nucleus (Top-p) Sampling', fontsize=13)
plt.tight_layout()
plt.show()

# Generate
prompt = torch.tensor([encode("the ")], dtype=torch.long)
for p in [0.5, 0.8, 0.95]:
    output = top_p_sample(model, prompt, p=p)
    print(f"top-p={p}: '{decode(output[0].tolist())}'")

## 6. Beam Search

Maintain top-$B$ candidate sequences. At each step, expand all candidates and keep the best $B$.

**Length normalization:** divide log-probability by $T^\alpha$ where $\alpha \in [0.6, 1.0]$ to avoid preferring short sequences.

In [ ]:
@torch.no_grad()
def beam_search(model, prompt_ids, max_new_tokens=60, beam_width=4, 
                length_penalty_alpha=0.7):
    """Beam search with length normalization."""
    model.eval()
    device = prompt_ids.device
    
    # Initialize beams: (sequence, log_prob)
    beams = [(prompt_ids[0].tolist(), 0.0)]  # single sequence
    
    for step in range(max_new_tokens):
        all_candidates = []
        
        for seq, score in beams:
            input_ids = torch.tensor([seq[-model.max_len:]], dtype=torch.long, device=device)
            logits = model(input_ids)[:, -1, :]  # (1, vocab)
            log_probs = F.log_softmax(logits, dim=-1)  # (1, vocab)
            
            # Get top-k candidates from this beam
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_width, dim=-1)
            
            for i in range(beam_width):
                new_seq = seq + [topk_ids[0, i].item()]
                new_score = score + topk_log_probs[0, i].item()
                all_candidates.append((new_seq, new_score))
        
        # Length normalization: score / len^alpha
        def normalized_score(seq_score):
            seq, score = seq_score
            gen_len = len(seq) - len(prompt_ids[0])
            if gen_len == 0:
                return score
            return score / (gen_len ** length_penalty_alpha)
        
        # Keep top beam_width candidates
        beams = sorted(all_candidates, key=normalized_score, reverse=True)[:beam_width]
    
    # Return best beam
    best_seq = max(beams, key=normalized_score)[0]
    return torch.tensor([best_seq], dtype=torch.long, device=device)

# Generate with beam search
prompt = torch.tensor([encode("the ")], dtype=torch.long)
for bw in [1, 3, 5]:
    output = beam_search(model, prompt, beam_width=bw)
    print(f"beam={bw}: '{decode(output[0].tolist())}'")

print("\nBeam=1 is equivalent to greedy decoding.")

## 7. Repetition Penalty (Keskar et al., 2019)

Reduce the logit of tokens that have already appeared:
$$z_i' = \begin{cases} z_i / \theta & \text{if } z_i > 0 \text{ and token } i \text{ was generated} \\ z_i \times \theta & \text{if } z_i \leq 0 \text{ and token } i \text{ was generated} \end{cases}$$

where $\theta > 1$ is the penalty factor.

In [ ]:
def apply_repetition_penalty(logits: torch.Tensor, generated_ids: list, 
                              penalty: float = 1.2) -> torch.Tensor:
    """Apply repetition penalty to logits."""
    logits = logits.clone()
    for token_id in set(generated_ids):
        if logits[0, token_id] > 0:
            logits[0, token_id] /= penalty
        else:
            logits[0, token_id] *= penalty
    return logits


@torch.no_grad()
def sample_with_repetition_penalty(model, prompt_ids, max_new_tokens=60, 
                                    temperature=1.0, top_p=0.9, rep_penalty=1.2):
    model.eval()
    ids = prompt_ids.clone()
    generated = []
    
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])[:, -1:, :]
        logits = logits / temperature
        
        # Apply repetition penalty
        logits = apply_repetition_penalty(logits.squeeze(1), generated, rep_penalty)
        logits = logits.unsqueeze(1)
        
        # Top-p filtering
        logits_filtered = top_p_filtering(logits.squeeze(1), top_p)
        probs = F.softmax(logits_filtered, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        
        ids = torch.cat([ids, next_id], dim=1)
        generated.append(next_id.item())
    
    return ids

prompt = torch.tensor([encode("the ")], dtype=torch.long)
for rep in [1.0, 1.2, 1.5, 2.0]:
    output = sample_with_repetition_penalty(model, prompt, rep_penalty=rep)
    print(f"rep_penalty={rep}: '{decode(output[0].tolist())}'")

## 8. Combined Strategy

In practice, strategies are combined. A typical configuration:
- `temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.1`

In [ ]:
@torch.no_grad()
def generate(
    model,
    prompt_ids: torch.Tensor,
    max_new_tokens: int = 60,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
    repetition_penalty: float = 1.0,
) -> torch.Tensor:
    """Full generation pipeline combining all strategies."""
    model.eval()
    ids = prompt_ids.clone()
    generated = []
    
    for _ in range(max_new_tokens):
        logits = model(ids[:, -model.max_len:])[:, -1, :]  # (B, V)
        
        # 1. Repetition penalty
        if repetition_penalty != 1.0:
            logits = apply_repetition_penalty(logits, generated, repetition_penalty)
        
        # 2. Temperature
        if temperature != 1.0:
            logits = logits / temperature
        
        # 3. Top-k filtering
        if top_k > 0:
            logits = top_k_filtering(logits, top_k)
        
        # 4. Top-p (nucleus) filtering
        if top_p < 1.0:
            logits = top_p_filtering(logits, top_p)
        
        # 5. Sample
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        
        ids = torch.cat([ids, next_id], dim=1)
        generated.append(next_id.item())
    
    return ids

# Compare strategies side by side
prompt = torch.tensor([encode("the ")], dtype=torch.long)

configs = [
    {'temperature': 0.01, 'label': 'Near-greedy (T=0.01)'},
    {'temperature': 0.7, 'top_p': 0.9, 'label': 'Balanced (T=0.7, p=0.9)'},
    {'temperature': 1.0, 'top_k': 5, 'label': 'Top-k=5 (T=1.0)'},
    {'temperature': 0.8, 'top_p': 0.9, 'repetition_penalty': 1.3, 
     'label': 'Full combo (T=0.8, p=0.9, rep=1.3)'},
]

for cfg in configs:
    label = cfg.pop('label')
    output = generate(model, prompt, **cfg)
    print(f"{label}:")
    print(f"  '{decode(output[0].tolist())}'\n")

## 9. Diversity Analysis

Generate multiple samples and measure diversity.

In [ ]:
def measure_diversity(model, prompt_ids, strategy_fn, n_samples=20, gen_len=40):
    """Generate n samples and measure diversity metrics."""
    samples = []
    for _ in range(n_samples):
        output = strategy_fn(model, prompt_ids, max_new_tokens=gen_len)
        text = decode(output[0].tolist())
        samples.append(text)
    
    # Unique samples
    n_unique = len(set(samples))
    
    # Unique trigrams across all samples
    all_trigrams = []
    for s in samples:
        for i in range(len(s) - 2):
            all_trigrams.append(s[i:i+3])
    n_unique_trigrams = len(set(all_trigrams))
    
    return n_unique, n_unique_trigrams, samples

prompt = torch.tensor([encode("the ")], dtype=torch.long)

strategies = {
    'Greedy': lambda m, p, **kw: greedy_decode(m, p, 40),
    'T=0.5': lambda m, p, **kw: sample_with_temperature(m, p, 40, temperature=0.5),
    'T=1.0': lambda m, p, **kw: sample_with_temperature(m, p, 40, temperature=1.0),
    'T=1.5': lambda m, p, **kw: sample_with_temperature(m, p, 40, temperature=1.5),
    'Top-p=0.9': lambda m, p, **kw: top_p_sample(m, p, 40, p=0.9),
    'Top-k=5': lambda m, p, **kw: top_k_sample(m, p, 40, k=5),
}

results = {}
for name, fn in strategies.items():
    n_uniq, n_tri, _ = measure_diversity(model, prompt, fn)
    results[name] = (n_uniq, n_tri)
    print(f"{name:>15}: {n_uniq:>3} unique / 20 samples, {n_tri:>4} unique trigrams")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(results.keys())
uniq_samples = [results[n][0] for n in names]
uniq_trigrams = [results[n][1] for n in names]

axes[0].bar(names, uniq_samples, color='steelblue')
axes[0].set_ylabel('Unique samples (out of 20)')
axes[0].set_title('Sample Diversity')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, uniq_trigrams, color='coral')
axes[1].set_ylabel('Unique trigrams')
axes[1].set_title('Trigram Diversity')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Interview Notes

**"Beam search vs sampling — when to use which?"**
- Beam search: deterministic, finds high-probability sequences. Good for translation, summarization, structured output.
- Sampling: stochastic, produces diverse outputs. Good for creative text, dialogue, brainstorming.
- Beam search suffers from "boring" outputs and degenerate repetition at high beam widths.
- Most modern LLM deployments use sampling (temperature + top-p) rather than beam search.

**"What does temperature do?"**
- Divides logits by $T$ before softmax: $p_i \propto \exp(z_i / T)$
- $T < 1$: sharpens distribution (more deterministic)
- $T > 1$: flattens distribution (more random)
- $T \to 0$: greedy, $T \to \infty$: uniform

**"Top-p vs top-k?"**
- Top-k: fixed number of candidates regardless of confidence
- Top-p: adaptive — uses fewer candidates when model is confident, more when uncertain
- Top-p is generally preferred (used in GPT-4, Claude) because it adapts to the distribution shape

**"What are common production settings?"**
- Code generation: `T=0.2, top_p=0.95` (low diversity, high accuracy)
- Chat: `T=0.7, top_p=0.9` (balanced)
- Creative writing: `T=1.0, top_p=0.95` (high diversity)
- Factual QA: `T=0.0` (greedy or beam search)